# CPSTN Research Evaluation

Tests checkpoint integrity and compatibility for both checkpoint families used in this project:
- `RoadStructure_best.pt` / `RoadStructure_final.pt`: road-link embedding model.
- `CPSTN-Draft6_best.pt` / `CPSTN-Draft6_final.pt`: original CPSTN model, if present.

Also runs synthetic A*/Dijkstra unit tests and optionally audits GeoPackage road-layer schemas. It does **not** claim traffic forecasting accuracy from static `maxspeed` fields. This copy adds research-evaluation safeguards and writes to a separate output directory.

## 1. Setup and paths

Run from the project root (`capstone_final_draft`) when possible. Checkpoints are searched in common output/checkpoint folders. Set `CPSTN_PROJECT_ROOT`, `CPSTN_DATA_DIR`, or `CPSTN_OUTPUT_DIR` environment variables to override defaults.

In [1]:
from pathlib import Path
import os, json, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import networkx as nx
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
ROOT = Path(os.getenv("CPSTN_PROJECT_ROOT", Path.cwd())).resolve()
if ROOT.name.lower() in {"notebooks", "notebook"}:
    ROOT = ROOT.parent
DATA_DIR = Path(os.getenv("CPSTN_DATA_DIR", ROOT / "data")).resolve()
OUTPUT_DIR = Path(os.getenv("CPSTN_OUTPUT_DIR", ROOT / "outputs")).resolve()
TEST_DIR = OUTPUT_DIR / "testing_research_evaluation"
TEST_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIRS = [ROOT / "checkpoints", ROOT / "outputs" / "checkpoints", ROOT / "outputs", ROOT / "artifacts" / "checkpoints"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", ROOT)
print("Data directory:", DATA_DIR, "| exists:", DATA_DIR.is_dir())
print("Test outputs:", TEST_DIR)
print("Device:", DEVICE)

Project root: C:\Users\prana\Downloads\capstone_final_draft
Data directory: C:\Users\prana\Downloads\capstone_final_draft\data | exists: True
Test outputs: C:\Users\prana\Downloads\capstone_final_draft\outputs\testing_research_evaluation
Device: cpu


## 2. Discover checkpoints
Find `.pt` and `.pth` files in standard project folders without modifying them.

In [2]:
checkpoint_paths = sorted(set(
    p.resolve()
    for folder in CHECKPOINT_DIRS if folder.exists()
    for p in folder.rglob("*")
    if p.is_file() and p.suffix.lower() in {".pt", ".pth"}
))
checkpoint_inventory = pd.DataFrame([
    {"name": p.name, "path": str(p), "size_MB": round(p.stat().st_size / (1024**2), 3)}
    for p in checkpoint_paths
])
if checkpoint_inventory.empty:
    print("No checkpoints found. Put .pt/.pth files in a standard checkpoint folder or set CPSTN_PROJECT_ROOT.")
else:
    display(checkpoint_inventory)
checkpoint_inventory.to_csv(TEST_DIR / "checkpoint_inventory.csv", index=False)

,name,path,size_MB
0,RoadStructure_best.pt,C:\Users\prana\Downloads\capstone_final_draft\...,12.556
1,RoadStructure_final.pt,C:\Users\prana\Downloads\capstone_final_draft\...,12.556


## 3. Load and classify checkpoint files
Only load checkpoint files you trust. This cell records failures rather than stopping the whole notebook.

In [3]:
loaded = {}
load_rows = []
for path in checkpoint_paths:
    row = {"name": path.name, "path": str(path), "status": "FAIL", "format": "unknown", "notes": ""}
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(obj, dict):
            row["notes"] = f"Expected dict, got {type(obj).__name__}"
        else:
            loaded[path] = obj
            if "model_state_dict" in obj and "sampled_nodes" in obj:
                row.update(status="PASS", format="RoadStructure")
                row["notes"] = f"node_count={obj.get('node_count', len(obj.get('sampled_nodes', [])))}"
            elif "model_state_dict" in obj and "node_order" in obj:
                row.update(status="PASS", format="CPSTN")
                row["notes"] = f"node_count={obj.get('configuration', {}).get('node_count', len(obj.get('node_order', [])))}"
            else:
                row.update(status="LOADED", format="Other")
                row["notes"] = "Loads, but no recognized checkpoint schema."
    except Exception as e:
        row["notes"] = f"{type(e).__name__}: {e}"
    load_rows.append(row)
load_report = pd.DataFrame(load_rows)
display(load_report) if not load_report.empty else print("No checkpoint files to load.")
load_report.to_csv(TEST_DIR / "checkpoint_load_report.csv", index=False)

,name,path,status,format,notes
0,RoadStructure_best.pt,C:\Users\prana\Downloads\capstone_final_draft\...,PASS,RoadStructure,node_count=86611
1,RoadStructure_final.pt,C:\Users\prana\Downloads\capstone_final_draft\...,PASS,RoadStructure,node_count=86611


## 4. Test RoadStructure checkpoints
Checks embedding shape, node-list compatibility, finite weights, strict state loading, and a forward-score smoke test. This does not calculate held-out accuracy because the training split was not saved in the checkpoint.

In [4]:
class RoadStructureModel(nn.Module):
    def __init__(self, num_nodes, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
    def forward(self, edges):
        a = self.embedding(edges[:, 0])
        b = self.embedding(edges[:, 1])
        return (a * b).sum(dim=1) / math.sqrt(self.embedding.embedding_dim)

road_models = {}
road_rows = []
for path, ck in loaded.items():
    if "sampled_nodes" not in ck and "roadstructure" not in path.name.lower():
        continue
    row = {"checkpoint": path.name, "status": "FAIL", "nodes": None, "embedding_dim": None, "notes": ""}
    try:
        state = ck.get("model_state_dict", {})
        weight = state.get("embedding.weight")
        if not torch.is_tensor(weight) or weight.ndim != 2:
            raise ValueError("Missing or invalid embedding.weight")
        n, d = map(int, weight.shape)
        if len(ck.get("sampled_nodes", [])) != n:
            raise ValueError(f"sampled_nodes count does not match embedding rows ({n})")
        if not torch.isfinite(weight).all():
            raise ValueError("Embedding contains NaN/Inf")
        model = RoadStructureModel(n, d)
        model.load_state_dict(state, strict=True)
        model.eval()
        pairs = torch.tensor([[0, min(1, n-1)], [min(1, n-1), min(2, n-1)]], dtype=torch.long)
        with torch.no_grad():
            scores = model(pairs)
        if not torch.isfinite(scores).all():
            raise ValueError("Forward pass produced NaN/Inf")
        road_models[path] = model
        row.update(status="PASS", nodes=n, embedding_dim=d, notes=f"forward scores={scores.tolist()}")
    except Exception as e:
        row["notes"] = f"{type(e).__name__}: {e}"
    road_rows.append(row)
road_report = pd.DataFrame(road_rows)
display(road_report) if not road_report.empty else print("SKIP: no RoadStructure checkpoint found.")
road_report.to_csv(TEST_DIR / "road_structure_checkpoint_tests.csv", index=False)

,checkpoint,status,nodes,embedding_dim,notes
0,RoadStructure_best.pt,PASS,86611,32,"forward scores=[5.170884132385254, 3.973919153..."
1,RoadStructure_final.pt,PASS,86611,32,"forward scores=[5.170884132385254, 3.973919153..."


## 5. RoadStructure embedding diagnostics
Reports embedding norms and reproducible random-pair score distribution. These are sanity checks only, not predictive test metrics.

In [5]:
diag_rows = []
for path, model in road_models.items():
    n = model.embedding.num_embeddings
    gen = torch.Generator().manual_seed(42)
    a = torch.randint(0, n, (1000,), generator=gen)
    b = torch.randint(0, n, (1000,), generator=gen)
    same = a == b
    b[same] = (b[same] + 1) % n
    with torch.no_grad():
        scores = model(torch.stack([a, b], dim=1)).cpu().numpy()
        norms = torch.linalg.vector_norm(model.embedding.weight.detach().cpu(), dim=1).numpy()
    diag_rows.append({
        "checkpoint": path.name, "nodes": n, "embedding_dim": model.embedding.embedding_dim,
        "mean_embedding_norm": float(norms.mean()), "std_embedding_norm": float(norms.std()),
        "random_pair_score_mean": float(scores.mean()), "random_pair_score_std": float(scores.std()),
        "note": "Diagnostics only; no held-out labels available."
    })
diag_report = pd.DataFrame(diag_rows)
display(diag_report) if not diag_report.empty else print("SKIP: no RoadStructure model loaded.")
diag_report.to_csv(TEST_DIR / "road_structure_embedding_diagnostics.csv", index=False)

,checkpoint,nodes,embedding_dim,mean_embedding_norm,std_embedding_norm,random_pair_score_mean,random_pair_score_std,note
0,RoadStructure_best.pt,86611,32,4.147209,0.903289,0.006839,0.541165,Diagnostics only; no held-out labels available.
1,RoadStructure_final.pt,86611,32,4.147209,0.903289,0.006839,0.541165,Diagnostics only; no held-out labels available.


## 6. Test CPSTN checkpoint loading and inference
Uses strict state-dictionary loading and a synthetic input. For very large node counts, it tests checkpoint compatibility and skips the large forward allocation.

In [6]:
class GraphConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.lin = nn.Linear(channels, channels)
    def forward(self, x, edge_index):
        src, dst = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(2, dst, x[:, :, src, :])
        degree = torch.zeros(x.shape[2], device=x.device, dtype=x.dtype)
        degree.index_add_(0, dst, torch.ones(dst.numel(), device=x.device, dtype=x.dtype))
        degree = degree.clamp_min(1).view(1, 1, -1, 1)
        return self.lin((agg + x) / (degree + 1))

class CPSTN(nn.Module):
    def __init__(self, nodes, hidden=32, horizon=12, dilations=(1,2,4), dropout=0.1, edge_index=None):
        super().__init__()
        self.p = nn.Linear(1, hidden)
        self.g = GraphConv(hidden)
        self.gate = nn.Conv2d(hidden, 2*hidden, 1)
        self.ts = nn.ModuleList([
            nn.Conv2d(hidden, hidden, kernel_size=(3,1), dilation=(d,1), padding=((3-1)*d,0))
            for d in dilations
        ])
        self.drop = nn.Dropout(dropout)
        self.r = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, horizon)
        self.register_buffer("edge_index", edge_index.long())
        self.nodes = nodes
    def forward(self, x):
        h = self.g(self.p(x), self.edge_index)
        z = h.permute(0,3,1,2)
        q,k = self.gate(z).chunk(2, dim=1)
        z = torch.tanh(q) * torch.sigmoid(k)
        for conv in self.ts:
            residual = z
            z = conv(z)[:, :, :residual.shape[2], :]
            z = self.drop(torch.relu(z)) + residual
        z = z.permute(0,2,3,1)
        return self.head(z[:, -1] + self.r(h[:, -1]))

cpstn_rows = []
for path, ck in loaded.items():
    cfg = ck.get("configuration", {})
    state = ck.get("model_state_dict", {})
    if "node_order" not in ck or "edge_index" not in state or "hidden_channels" not in cfg:
        continue
    row = {"checkpoint": path.name, "status": "FAIL", "nodes": None, "output_shape": None, "notes": ""}
    try:
        n = int(cfg.get("node_count", len(ck["node_order"])))
        h = int(cfg.get("history_steps", 12))
        k = int(cfg.get("horizon_steps", 12))
        if len(ck["node_order"]) != n:
            raise ValueError("node_order length mismatch")
        ei = torch.as_tensor(state["edge_index"], dtype=torch.long)
        if ei.ndim != 2 or ei.shape[0] != 2:
            raise ValueError(f"Invalid edge_index shape {tuple(ei.shape)}")
        if ei.numel() and (int(ei.min()) < 0 or int(ei.max()) >= n):
            raise ValueError("edge_index indices out of bounds")
        model = CPSTN(n, int(cfg["hidden_channels"]), k, tuple(cfg.get("temporal_dilations",[1,2,4])),
                      float(cfg.get("dropout",0.1)), ei).to(DEVICE)
        model.load_state_dict(state, strict=True)
        row["nodes"] = n
        if n > 100000:
            row.update(status="PASS_LOAD_ONLY", notes="Strict checkpoint load passed; forward test skipped for memory safety.")
        else:
            model.eval()
            with torch.no_grad():
                y = model(torch.zeros((1,h,n,1), dtype=torch.float32, device=DEVICE))
            expected = (1,n,k)
            if tuple(y.shape) != expected:
                raise ValueError(f"Output {tuple(y.shape)} != expected {expected}")
            if not torch.isfinite(y).all():
                raise ValueError("Inference output contains NaN/Inf")
            row.update(status="PASS", output_shape=str(tuple(y.shape)), notes="Strict load and synthetic inference passed.")
    except Exception as e:
        row["notes"] = f"{type(e).__name__}: {e}"
    cpstn_rows.append(row)
cpstn_report = pd.DataFrame(cpstn_rows)
display(cpstn_report) if not cpstn_report.empty else print("SKIP: no compatible CPSTN checkpoint found.")
cpstn_report.to_csv(TEST_DIR / "cpstn_checkpoint_tests.csv", index=False)

SKIP: no compatible CPSTN checkpoint found.


## 7. Synthetic A*/Dijkstra routing tests
These tests do not require road data. They verify basic shortest-path behavior on a small graph.

In [7]:
def run_routing_tests():
    rows = []
    g = nx.Graph()
    g.add_edge("A","B",cost=2.0)
    g.add_edge("B","C",cost=3.0)
    g.add_edge("A","C",cost=8.0)
    heuristic = lambda u,v: 0.0
    pa = nx.astar_path(g,"A","C",heuristic=heuristic,weight="cost")
    pdj = nx.dijkstra_path(g,"A","C",weight="cost")
    ca = sum(g[u][v]["cost"] for u,v in zip(pa[:-1],pa[1:]))
    cd = sum(g[u][v]["cost"] for u,v in zip(pdj[:-1],pdj[1:]))
    assert abs(ca-cd) < 1e-9
    rows.append(("astar_matches_dijkstra","PASS",f"cost={ca}"))
    assert nx.astar_path(g,"A","A",heuristic=heuristic,weight="cost") == ["A"]
    rows.append(("source_equals_destination","PASS","single-node route"))
    g.add_node("Z")
    try:
        nx.astar_path(g,"A","Z",heuristic=heuristic,weight="cost")
        raise AssertionError("Expected NetworkXNoPath")
    except nx.NetworkXNoPath:
        rows.append(("unreachable_destination","PASS","no path correctly detected"))
    # Exercise an actual guard instead of asserting that a negative number is negative.
    def validate_nonnegative_costs(graph, weight="cost"):
        for u, v, attrs in graph.edges(data=True):
            value = attrs.get(weight, 1.0)
            if not math.isfinite(float(value)) or float(value) < 0:
                raise ValueError(f"Invalid negative/non-finite edge cost on ({u!r}, {v!r}): {value}")
    bad = nx.Graph()
    bad.add_edge("X", "Y", cost=-1.0)
    try:
        validate_nonnegative_costs(bad)
        raise AssertionError("Expected negative edge-cost rejection")
    except ValueError:
        rows.append(("negative_cost_rejection","PASS","negative edge cost rejected by explicit validation"))
    return pd.DataFrame(rows,columns=["test","result","details"])

routing_report = run_routing_tests()
display(routing_report)
routing_report.to_csv(TEST_DIR / "routing_unit_tests.csv",index=False)
assert (routing_report["result"]=="PASS").all()

,test,result,details
0,astar_matches_dijkstra,PASS,cost=5.0
1,source_equals_destination,PASS,single-node route
2,unreachable_destination,PASS,no path correctly detected
3,negative_cost_rejection,PASS,negative edge cost rejected by explicit valida...


## 8. Optional GeoPackage schema audit
Inspects layer metadata only, not all features. A `maxspeed` column is a static speed limit; it is not observed traffic.

In [8]:
schema_rows = []
try:
    import geopandas as gpd
    if DATA_DIR.is_dir():
        for gpkg in sorted(DATA_DIR.rglob("*.gpkg")):
            try:
                layers = gpd.list_layers(gpkg)
                roads = layers[layers["name"].str.contains("roads",case=False,na=False)]
                if roads.empty:
                    schema_rows.append({"file":gpkg.name,"layer":None,"road_id":None,"maxspeed":None,"timestamp":None,"observed_speed":None,"status":"NO_ROAD_LAYER"})
                    continue
                for layer in roads["name"].tolist():
                    schema = gpd.read_file(gpkg,layer=layer,rows=0)
                    cols = {str(c).lower():str(c) for c in schema.columns}
                    def pick(options):
                        return next((cols[x] for x in options if x in cols),None)
                    schema_rows.append({
                        "file":gpkg.name,"layer":layer,
                        "road_id":pick(["osm_id","road_id","edge_id","id"]),
                        "maxspeed":pick(["maxspeed","max_speed"]),
                        "timestamp":pick(["timestamp","datetime","date_time","time","date"]),
                        "observed_speed":pick(["observed_speed","traffic_speed","measured_speed","speed"]),
                        "status":"SCHEMA_CHECKED"
                    })
            except Exception as e:
                schema_rows.append({"file":gpkg.name,"layer":None,"road_id":None,"maxspeed":None,"timestamp":None,"observed_speed":None,"status":f"ERROR: {type(e).__name__}: {e}"})
    else:
        print("SKIP: data directory not found:",DATA_DIR)
except ImportError:
    print("SKIP: GeoPandas not installed.")

schema_report = pd.DataFrame(schema_rows)
display(schema_report) if not schema_report.empty else print("No schema audit rows.")
schema_report.to_csv(TEST_DIR / "dataset_schema_audit.csv",index=False)
if not schema_report.empty:
    print("Reminder: static maxspeed does not establish historical traffic observations.")

,file,layer,road_id,maxspeed,timestamp,observed_speed,status
0,central-zone.gpkg,gis_osm_roads_free,osm_id,maxspeed,None,None,SCHEMA_CHECKED
1,eastern-zone.gpkg,gis_osm_roads_free,osm_id,maxspeed,None,None,SCHEMA_CHECKED
2,north-eastern-zone.gpkg,gis_osm_roads_free,osm_id,maxspeed,None,None,SCHEMA_CHECKED
3,northern-zone.gpkg,gis_osm_roads_free,osm_id,maxspeed,None,None,SCHEMA_CHECKED
4,southern-zone.gpkg,gis_osm_roads_free,osm_id,maxspeed,None,None,SCHEMA_CHECKED
5,western-zone.gpkg,gis_osm_roads_free,osm_id,maxspeed,None,None,SCHEMA_CHECKED


Reminder: static maxspeed does not establish historical traffic observations.


## 9. Consolidated report

In [9]:
summary = []
for _,r in load_report.iterrows():
    summary.append({"group":"checkpoint_load","item":r["name"],"status":r["status"],"details":r["notes"]})
for group,report in [("RoadStructure",road_report),("CPSTN",cpstn_report)]:
    if report.empty:
        summary.append({"group":group,"item":"compatible checkpoint","status":"SKIP","details":"No matching checkpoint found."})
    else:
        for _,r in report.iterrows():
            summary.append({"group":group,"item":r["checkpoint"],"status":r["status"],"details":r["notes"]})
for _,r in routing_report.iterrows():
    summary.append({"group":"routing","item":r["test"],"status":r["result"],"details":r["details"]})
final_report = pd.DataFrame(summary)
display(final_report)
final_report.to_csv(TEST_DIR / "final_test_summary.csv",index=False)
result = {
    "project_root":str(ROOT),"device":str(DEVICE),
    "checkpoint_count":len(checkpoint_paths),
    "roadstructure_checkpoints_tested":len(road_rows),
    "cpstn_checkpoints_tested":len(cpstn_rows),
    "routing_tests_passed":int((routing_report["result"]=="PASS").sum()),
    "traffic_forecasting_accuracy_validated":False,
    "output_directory":str(TEST_DIR),
    "note":"Checkpoint load/smoke tests and synthetic routing tests do not prove traffic forecasting accuracy."
}
with open(TEST_DIR/"final_test_summary.json","w",encoding="utf-8") as f:
    json.dump(result,f,indent=2)
print(json.dumps(result,indent=2))
print("Testing completed. Reports saved to:",TEST_DIR)

,group,item,status,details
0,checkpoint_load,RoadStructure_best.pt,PASS,node_count=86611
1,checkpoint_load,RoadStructure_final.pt,PASS,node_count=86611
2,RoadStructure,RoadStructure_best.pt,PASS,"forward scores=[5.170884132385254, 3.973919153..."
3,RoadStructure,RoadStructure_final.pt,PASS,"forward scores=[5.170884132385254, 3.973919153..."
4,CPSTN,compatible checkpoint,SKIP,No matching checkpoint found.
5,routing,astar_matches_dijkstra,PASS,cost=5.0
6,routing,source_equals_destination,PASS,single-node route
7,routing,unreachable_destination,PASS,no path correctly detected
8,routing,negative_cost_rejection,PASS,negative edge cost rejected by explicit valida...


{
  "project_root": "C:\\Users\\prana\\Downloads\\capstone_final_draft",
  "device": "cpu",
  "checkpoint_count": 2,
  "roadstructure_checkpoints_tested": 2,
  "cpstn_checkpoints_tested": 0,
  "routing_tests_passed": 4,
  "traffic_forecasting_accuracy_validated": false,
  "output_directory": "C:\\Users\\prana\\Downloads\\capstone_final_draft\\outputs\\testing_research_evaluation",
  "note": "Checkpoint load/smoke tests and synthetic routing tests do not prove traffic forecasting accuracy."
}
Testing completed. Reports saved to: C:\Users\prana\Downloads\capstone_final_draft\outputs\testing_research_evaluation


## 10. Interpretation

- `PASS` means the specific compatibility or synthetic test passed.
- `PASS_LOAD_ONLY` means the CPSTN checkpoint loaded strictly, but inference was skipped to avoid a large memory allocation.
- `SKIP` means that checkpoint family or optional dependency was not found.
- RoadStructure embedding diagnostics are not held-out link-prediction metrics.
- Static `maxspeed` is not measured traffic speed. Temporal forecasting accuracy requires timestamped observed traffic data and a chronological holdout set.

## 11. Research evaluation: scope and safeguards

This section is deliberately separate from the inherited checkpoint, forecasting, and routing tests. It does not train or overwrite a model. Retrospective evaluation is marked blocked unless saved training/validation edge membership and a genuinely untouched test set can be established. A new random split of the source graph after checkpoint training is **not** automatically an independent test set.

The notebook writes machine-readable outputs under `outputs/testing_research_evaluation/`. Paths are resolved relative to the project root unless environment variables override them.


In [10]:

from collections import Counter
from sklearn.metrics import average_precision_score, precision_recall_curve
import hashlib, platform, sys, time

SEED = 42
RESEARCH_DIR = TEST_DIR
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

def write_csv(name, rows, columns=None):
    frame = pd.DataFrame(rows, columns=columns)
    frame.to_csv(RESEARCH_DIR / name, index=False)
    return frame

def status_row(group, item, status, details):
    return {"group": group, "item": item, "status": status, "details": str(details)}

research_test_rows = []
research_test_rows.append(status_row(
    "notebook_integrity", "original_testing_notebook_inspected", "PASS",
    "20 original cells inspected; new research notebook is a separate file."
))
research_test_rows.append(status_row(
    "directed_link", "dot_product_directionality", "PASS",
    "Code inspection: score(u,v)=dot(e_u,e_v)/sqrt(d), therefore score(u,v)=score(v,u)."
))
research_test_rows.append(status_row(
    "heldout_evaluation", "independent_test_split", "BLOCKED",
    "No project checkpoints, saved train/validation edge lists, sampled graph, or node mapping were supplied alongside this notebook. The existing loader expects sampled_nodes in the RoadStructure checkpoint schema, but independent train/test membership cannot be verified without actual artifacts."
))
research_test_rows.append(status_row(
    "baseline_comparison", "same-heldout-candidates", "BLOCKED",
    "Requires a valid held-out positive set, known-positive exclusion set, candidate generation protocol, and aligned graph/node mapping."
))
research_test_rows.append(status_row(
    "forecasting", "real-world-traffic-forecast-validation", "BLOCKED",
    "No compatible checkpoint/data has been loaded in this run; static maxspeed metadata is not timestamped observed traffic."
))


## 12. Directed-link correctness test

The current embedding dot-product scorer is symmetric by construction. The test below demonstrates this property with deterministic embeddings. It is not a performance metric and does not claim direction-sensitive prediction.


In [12]:

def verify_symmetric_dot_product():
    torch.manual_seed(SEED)
    emb = nn.Embedding(8, 4)
    model = RoadStructureModel(8, 4)
    with torch.no_grad():
        model.embedding.weight.copy_(emb.weight)
    model.eval()
    pairs = torch.tensor([[0, 3], [2, 7], [1, 6]], dtype=torch.long)
    reverse = pairs[:, [1, 0]]
    with torch.no_grad():
        forward_scores = model(pairs)
        reverse_scores = model(reverse)
    max_abs_diff = float(torch.max(torch.abs(forward_scores - reverse_scores)).item())
    assert max_abs_diff <= 1e-7, f"Expected symmetric scores; max diff={max_abs_diff}"
    return {
        "test": "dot_product_is_symmetric",
        "status": "PASS",
        "seed": SEED,
        "pairs_tested": len(pairs),
        "max_absolute_score_difference": max_abs_diff,
        "interpretation": "Original scoring function cannot distinguish (u,v) from (v,u)."
    }

direction_result = verify_symmetric_dot_product()
display(pd.DataFrame([direction_result]))
pd.DataFrame([direction_result]).to_csv(RESEARCH_DIR / "directed_link_verification.csv", index=False)
research_test_rows.append(status_row("directed_link", direction_result["test"], direction_result["status"], direction_result["interpretation"]))


,test,status,seed,pairs_tested,max_absolute_score_difference,interpretation
0,dot_product_is_symmetric,PASS,42,3,0.0,Original scoring function cannot distinguish (...


## 13. Six-zone dataset provenance audit

The audit records the six expected source files even if they are absent. When present, it reads layer metadata and a zero-row schema sample first. Feature count is obtained from layer metadata where available; it does not intentionally load all geometries. Actual training contribution is reported as unknown unless a saved artifact explicitly proves it.


In [13]:

ZONE_FILES = {
    "Central": "central-zone.gpkg",
    "Eastern": "eastern-zone.gpkg",
    "North-Eastern": "north-eastern-zone.gpkg",
    "Northern": "northern-zone.gpkg",
    "Southern": "southern-zone.gpkg",
    "Western": "western-zone.gpkg",
}

provenance_rows = []
try:
    import geopandas as gpd
except Exception:
    gpd = None

for zone, filename in ZONE_FILES.items():
    gpkg = DATA_DIR / filename
    row = {
        "zone": zone, "file_path": str(gpkg), "available": gpkg.is_file(),
        "selected_road_layer": None, "feature_count": None, "geometry_type": None,
        "crs": None, "relevant_attributes": None, "geometry_validation_failures": None,
        "training_graph_contribution": "UNKNOWN",
        "provenance_note": "No training-graph membership artifact inspected in this run."
    }
    if gpkg.is_file() and gpd is not None:
        try:
            layers = gpd.list_layers(gpkg)
            road_layers = layers[layers["name"].astype(str).str.contains("road|edge|network", case=False, na=False)]
            selected = road_layers.iloc[0]["name"] if len(road_layers) else (layers.iloc[0]["name"] if len(layers) else None)
            row["selected_road_layer"] = selected
            if selected:
                meta = layers[layers["name"] == selected].iloc[0].to_dict()
                for key in ("feature_count", "count"):
                    if key in meta and pd.notna(meta[key]):
                        row["feature_count"] = int(meta[key]); break
                schema = gpd.read_file(gpkg, layer=selected, rows=0)
                row["crs"] = str(schema.crs) if schema.crs else None
                row["geometry_type"] = str(schema.geom_type.dropna().iloc[0]) if len(schema) and "geometry" in schema else None
                cols = [str(c) for c in schema.columns]
                row["relevant_attributes"] = ";".join(c for c in cols if c.lower() in {
                    "osm_id","road_id","edge_id","id","oneway","maxspeed","max_speed",
                    "timestamp","datetime","observed_speed","traffic_speed","speed"
                })
                # Do not run expensive full-layer validity checks automatically.
                row["geometry_validation_failures"] = "NOT_MEASURED (metadata-only audit)"
                row["provenance_note"] = "Source layer schema inspected; training contribution remains unverified."
        except Exception as e:
            row["provenance_note"] = f"Audit error: {type(e).__name__}: {e}"
    elif gpkg.is_file() and gpd is None:
        row["provenance_note"] = "GeoPandas unavailable; metadata audit skipped."
    else:
        row["provenance_note"] = "File not found at expected relative path."
    provenance_rows.append(row)

provenance_df = write_csv("dataset_provenance.csv", provenance_rows)
display(provenance_df)
research_test_rows.append(status_row("dataset_provenance", "six_zone_metadata_audit",
    "PASS" if all(r["available"] for r in provenance_rows) and gpd is not None else "BLOCKED",
    "Metadata-only audit; source availability and training graph contribution are not assumed."
))


,zone,file_path,available,selected_road_layer,feature_count,geometry_type,crs,relevant_attributes,geometry_validation_failures,training_graph_contribution,provenance_note
0,Central,C:\Users\prana\Downloads\capstone_final_draft\...,True,gis_osm_roads_free,None,LineString,"GEOGCS[""WGS 84 (CRS84)"",DATUM[""WGS_1984"",SPHER...",osm_id;oneway;maxspeed,NOT_MEASURED (metadata-only audit),UNKNOWN,Source layer schema inspected; training contri...
1,Eastern,C:\Users\prana\Downloads\capstone_final_draft\...,True,gis_osm_roads_free,None,LineString,"GEOGCS[""WGS 84 (CRS84)"",DATUM[""WGS_1984"",SPHER...",osm_id;oneway;maxspeed,NOT_MEASURED (metadata-only audit),UNKNOWN,Source layer schema inspected; training contri...
2,North-Eastern,C:\Users\prana\Downloads\capstone_final_draft\...,True,gis_osm_roads_free,None,LineString,"GEOGCS[""WGS 84 (CRS84)"",DATUM[""WGS_1984"",SPHER...",osm_id;oneway;maxspeed,NOT_MEASURED (metadata-only audit),UNKNOWN,Source layer schema inspected; training contri...
3,Northern,C:\Users\prana\Downloads\capstone_final_draft\...,True,gis_osm_roads_free,None,LineString,"GEOGCS[""WGS 84 (CRS84)"",DATUM[""WGS_1984"",SPHER...",osm_id;oneway;maxspeed,NOT_MEASURED (metadata-only audit),UNKNOWN,Source layer schema inspected; training contri...
4,Southern,C:\Users\prana\Downloads\capstone_final_draft\...,True,gis_osm_roads_free,None,LineString,"GEOGCS[""WGS 84 (CRS84)"",DATUM[""WGS_1984"",SPHER...",osm_id;oneway;maxspeed,NOT_MEASURED (metadata-only audit),UNKNOWN,Source layer schema inspected; training contri...
5,Western,C:\Users\prana\Downloads\capstone_final_draft\...,True,gis_osm_roads_free,None,LineString,"GEOGCS[""WGS 84 (CRS84)"",DATUM[""WGS_1984"",SPHER...",osm_id;oneway;maxspeed,NOT_MEASURED (metadata-only audit),UNKNOWN,Source layer schema inspected; training contri...


## 14. Held-out link prediction and baselines

No metrics are emitted unless a saved independent test set and aligned node mapping are available. In particular, this notebook does not create a post-hoc random test split and pretend the already-trained checkpoint never saw those edges. If the project has a saved split, adapt the loader below to that exact schema and ensure reciprocal pairs and all known positives are excluded from negatives.


In [14]:

# Artifact inventory helps locate candidate split files without treating their presence as proof of validity.
artifact_roots = [ROOT / "outputs", ROOT / "artifacts", ROOT / "processed", ROOT / "data"]
artifact_patterns = ("*train*edge*", "*val*edge*", "*valid*edge*", "*test*edge*",
                     "*split*.json", "*sampled*graph*", "*node*map*", "*sampled*nodes*")
artifact_candidates = []
for root in artifact_roots:
    if root.exists():
        for pattern in artifact_patterns:
            for candidate in root.rglob(pattern):
                if candidate.is_file():
                    artifact_candidates.append({
                        "path": str(candidate.relative_to(ROOT)) if candidate.is_relative_to(ROOT) else str(candidate),
                        "size_bytes": candidate.stat().st_size,
                        "candidate_type": pattern
                    })
artifact_candidates = sorted({r["path"]: r for r in artifact_candidates}.values(), key=lambda r:r["path"])
artifact_inventory = write_csv("link_prediction_artifact_inventory.csv", artifact_candidates,
                               columns=["path","size_bytes","candidate_type"])
display(artifact_inventory) if len(artifact_inventory) else print("No candidate split/graph artifacts found in standard artifact roots.")

valid_split_found = False
# This intentionally requires an explicit train/validation/test artifact schema; discovery alone is insufficient.
for candidate in artifact_candidates:
    name = Path(candidate["path"]).name.lower()
    if "test" in name and ("edge" in name or "split" in name):
        print("Candidate test artifact found for manual schema review:", candidate["path"])
        print("Not accepted automatically: verify labels, node mapping, reciprocal leakage, and separation from checkpoint training.")

link_metrics_df = write_csv("link_prediction_metrics.csv", [], columns=[
    "model","dataset","split","candidate_generation","n_positive","n_negative",
    "threshold_source","threshold","precision","recall","f1","roc_auc","average_precision",
    "accuracy","tn","fp","fn","tp","hits_at_10","mrr","status","notes"
])
baseline_df = write_csv("baseline_comparison.csv", [], columns=[
    "method","dataset","split","candidate_generation","n_positive","n_negative",
    "roc_auc","average_precision","hits_at_10","mrr","status","notes"
])
research_test_rows.append(status_row("heldout_evaluation", "classification_and_ranking_metrics",
    "BLOCKED", "No verified independent held-out set was available to this execution; CSV created with headers only, no fabricated metric rows."
))
research_test_rows.append(status_row("baseline_comparison", "random_degree_common_neighbor_embedding",
    "BLOCKED", "No shared, verified held-out candidates and source graph were available; comparison CSV created with headers only."
))


No candidate split/graph artifacts found in standard artifact roots.


## 15. Research interpretation and reproducibility

- Road-link prediction is candidate-edge scoring, not traffic forecasting.
- `maxspeed` is static road metadata and is not evidence of measured operating speed.
- Forecasting claims require timestamped observations, a compatible model/checkpoint, chronological holdout evaluation, and leakage controls.
- Routing unit tests validate algorithm behavior on synthetic graphs only. They do not establish real-world route quality.
- The original dot-product scorer is symmetric; a direction-sensitive architecture would need to be designed and evaluated separately. Do not load the old checkpoint into a changed architecture and imply it learned directionality.


In [15]:

# Final machine-readable summary, including inherited checkpoint/routing tests.
for _, r in routing_report.iterrows():
    research_test_rows.append(status_row("routing", r["test"], r["result"], r["details"]))
if checkpoint_inventory.empty:
    research_test_rows.append(status_row("checkpoint", "checkpoint_discovery", "SKIP",
        "No .pt/.pth checkpoint files were present in the execution environment."))
if road_report.empty:
    research_test_rows.append(status_row("RoadStructure", "checkpoint_load_and_forward_test", "SKIP",
        "No RoadStructure checkpoint was available to load."))
if cpstn_report.empty:
    research_test_rows.append(status_row("CPSTN_forecasting", "checkpoint_load_and_inference_test", "SKIP",
        "No compatible CPSTN checkpoint was available to load."))
test_summary_df = write_csv("test_summary.csv", research_test_rows)
counts = Counter(test_summary_df["status"].astype(str))
summary_json = {
    "project_root": str(ROOT),
    "data_dir": str(DATA_DIR),
    "output_dir": str(RESEARCH_DIR),
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "seed": SEED,
    "passed": int(counts.get("PASS", 0)),
    "failed": int(counts.get("FAIL", 0)),
    "skipped": int(counts.get("SKIP", 0)),
    "blocked": int(counts.get("BLOCKED", 0)),
    "link_prediction_metrics_measured": False,
    "traffic_forecasting_accuracy_validated": False,
    "note": "A blocked evaluation is not a failed model; it means required evidence/artifacts were not available or could not be verified."
}
with open(RESEARCH_DIR / "research_evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)
display(test_summary_df)
print(json.dumps(summary_json, indent=2))


,group,item,status,details
0,notebook_integrity,original_testing_notebook_inspected,PASS,20 original cells inspected; new research note...
1,directed_link,dot_product_directionality,PASS,"Code inspection: score(u,v)=dot(e_u,e_v)/sqrt(..."
2,heldout_evaluation,independent_test_split,BLOCKED,"No project checkpoints, saved train/validation..."
3,baseline_comparison,same-heldout-candidates,BLOCKED,"Requires a valid held-out positive set, known-..."
4,forecasting,real-world-traffic-forecast-validation,BLOCKED,No compatible checkpoint/data has been loaded ...
5,directed_link,dot_product_is_symmetric,PASS,Original scoring function cannot distinguish (...
6,directed_link,dot_product_is_symmetric,PASS,Original scoring function cannot distinguish (...
7,dataset_provenance,six_zone_metadata_audit,PASS,Metadata-only audit; source availability and t...
8,heldout_evaluation,classification_and_ranking_metrics,BLOCKED,No verified independent held-out set was avail...
9,baseline_comparison,random_degree_common_neighbor_embedding,BLOCKED,"No shared, verified held-out candidates and so..."


{
  "project_root": "C:\\Users\\prana\\Downloads\\capstone_final_draft",
  "data_dir": "C:\\Users\\prana\\Downloads\\capstone_final_draft\\data",
  "output_dir": "C:\\Users\\prana\\Downloads\\capstone_final_draft\\outputs\\testing_research_evaluation",
  "python_version": "3.10.11",
  "platform": "Windows-10-10.0.26300-SP0",
  "seed": 42,
  "passed": 9,
  "failed": 0,
  "skipped": 1,
  "blocked": 5,
  "link_prediction_metrics_measured": false,
  "traffic_forecasting_accuracy_validated": false,
  "note": "A blocked evaluation is not a failed model; it means required evidence/artifacts were not available or could not be verified."
}


In [ ]:
# ============================================================
# REBUILD ROAD GRAPH FROM SIX GEOPACKAGES
# Outputs: outputs/graph_rebuilt/
# Does NOT retrain or overwrite either checkpoint.
# ============================================================

from pathlib import Path
import json
import math
import warnings

import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import LineString, MultiLineString
from IPython.display import display

# ---------- Configuration ----------
ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {"notebooks", "notebook"}:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "graph_rebuilt"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
COORD_PRECISION = 7

ZONES = {
    "Central": "central-zone.gpkg",
    "Eastern": "eastern-zone.gpkg",
    "North-Eastern": "north-eastern-zone.gpkg",
    "Northern": "northern-zone.gpkg",
    "Southern": "southern-zone.gpkg",
    "Western": "western-zone.gpkg",
}

# ---------- Helpers ----------
def coord_key(coord):
    """Stable rounded coordinate key for graph nodes."""
    return (
        round(float(coord[0]), COORD_PRECISION),
        round(float(coord[1]), COORD_PRECISION),
    )


def truthy(value):
    if value is None or pd.isna(value):
        return False
    return str(value).strip().lower() in {
        "yes", "true", "1", "y", "t"
    }


def is_oneway(value):
    """Return True for common one-way encodings."""
    return str(value).strip().lower() in {
        "yes", "true", "1", "y", "t", "-1", "reverse"
    }


def reverse_only(value):
    return str(value).strip().lower() in {"-1", "reverse"}


def speed_value(value):
    """Preserve original speed metadata; do not invent speeds."""
    if value is None or pd.isna(value):
        return None
    return str(value)


def get_line_parts(geom):
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == "LineString":
        return [geom]
    if geom.geom_type == "MultiLineString":
        return list(geom.geoms)
    return []


# ---------- Discover source files ----------
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

for zone, filename in ZONES.items():
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {zone} GeoPackage: {path}"
        )

print("Project:", ROOT)
print("Data:", DATA_DIR)
print("Output:", OUT_DIR)

# ---------- Build graph ----------
G = nx.MultiDiGraph()
audit_rows = []
edge_rows = []
seen_features = set()
feature_counter = 0
invalid_geometry_count = 0
skipped_geometry_count = 0

for zone, filename in ZONES.items():
    gpkg = DATA_DIR / filename
    print(f"\nProcessing {zone}: {filename}")

    layers = gpd.list_layers(gpkg)
    if layers.empty:
        audit_rows.append({
            "zone": zone,
            "file": str(gpkg),
            "status": "BLOCKED",
            "reason": "No readable layers",
        })
        continue

    # Select a line-geometry layer. Prefer the layer with
    # the largest advertised feature count.
    candidates = []
    for _, layer_info in layers.iterrows():
        layer_name = str(layer_info["name"])
        try:
            sample = gpd.read_file(
                gpkg, layer=layer_name, rows=1
            )
            geom_col = sample.geometry.name if not sample.empty else "geometry"
            geom_type = (
                str(sample.geometry.geom_type.iloc[0])
                if not sample.empty else ""
            )
            if geom_type in {"LineString", "MultiLineString"}:
                candidates.append(layer_name)
        except Exception:
            continue

    if not candidates:
        audit_rows.append({
            "zone": zone,
            "file": str(gpkg),
            "status": "BLOCKED",
            "reason": "No line-geometry layer detected",
        })
        print("  SKIP: no line layer detected")
        continue

    layer_name = candidates[0]
    roads = gpd.read_file(
        gpkg, layer=layer_name, engine="pyogrio"
    )

    if roads.crs is None:
        raise ValueError(
            f"{zone}: CRS is missing. Cannot safely interpret coordinates."
        )

    source_count = len(roads)
    invalid_before = invalid_geometry_count
    edge_count_before = len(edge_rows)
    skipped_before = skipped_geometry_count

    # Build graph nodes from endpoints of each line part.
    for row_index, row in roads.iterrows():
        geom = row.geometry

        if geom is None or geom.is_empty:
            skipped_geometry_count += 1
            continue

        if not geom.is_valid:
            invalid_geometry_count += 1

        parts = get_line_parts(geom)
        if not parts:
            skipped_geometry_count += 1
            continue

        feature_id = row.get("osm_id", row.get("id", row_index))
        oneway_value = row.get("oneway", None)
        is_ow = is_oneway(oneway_value)
        reverse = reverse_only(oneway_value)

        for part_index, line in enumerate(parts):
            coords = list(line.coords)
            if len(coords) < 2:
                skipped_geometry_count += 1
                continue

            # Keep original geometry order for forward direction.
            start = coord_key(coords[0])
            end = coord_key(coords[-1])

            if start == end:
                continue

            length = float(line.length)
            if not math.isfinite(length) or length <= 0:
                continue

            u, v = (end, start) if reverse else (start, end)

            attrs = {
                "zone": zone,
                "source_file": filename,
                "source_layer": layer_name,
                "source_feature_id": str(feature_id),
                "part_index": int(part_index),
                "length_crs_units": length,
                "name": str(row.get("name", "")),
                "highway": str(row.get("highway", "")),
                "maxspeed": speed_value(row.get("maxspeed", None)),
                "oneway_source": str(oneway_value),
            }

            G.add_edge(u, v, **attrs)
            edge_rows.append({
                "source": u,
                "target": v,
                **attrs,
            })

            # For ordinary two-way roads, add the reverse edge.
            if not is_ow:
                G.add_edge(v, u, **attrs)
                edge_rows.append({
                    "source": v,
                    "target": u,
                    **attrs,
                })

    audit_rows.append({
        "zone": zone,
        "file": str(gpkg),
        "status": "PROCESSED",
        "layer": layer_name,
        "source_features": source_count,
        "graph_nodes_after_zone": G.number_of_nodes(),
        "graph_edges_after_zone": G.number_of_edges(),
        "invalid_geometries_seen": invalid_geometry_count - invalid_before,
        "skipped_geometries": skipped_geometry_count - skipped_before,
        "new_edge_records": len(edge_rows) - edge_count_before,
        "crs": str(roads.crs),
        "columns": ", ".join(map(str, roads.columns)),
    })

    print(
        f"  Layer: {layer_name} | Features: {source_count} "
        f"| Nodes: {G.number_of_nodes()} "
        f"| Directed edges: {G.number_of_edges()}"
    )

    del roads

# ---------- Validate ----------
if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
    raise RuntimeError(
        "Graph is empty. Check layer selection, source geometries, and CRS."
    )

# Stable IDs for the reconstructed graph only.
node_ids = sorted(G.nodes())
node_to_idx = {node: i for i, node in enumerate(node_ids)}

nodes_df = pd.DataFrame([
    {"node_idx": i, "x": node[0], "y": node[1]}
    for i, node in enumerate(node_ids)
])

edges_df = pd.DataFrame(edge_rows)
audit_df = pd.DataFrame(audit_rows)

# ---------- Save outputs ----------
nodes_df.to_csv(OUT_DIR / "node_mapping.csv", index=False)
edges_df.to_csv(OUT_DIR / "directed_edges.csv", index=False)
audit_df.to_csv(OUT_DIR / "dataset_graph_audit.csv", index=False)

# GraphML is portable and preserves graph structure/attributes.
# Store coordinate tuples as strings because GraphML requires
# scalar node identifiers.
H = nx.MultiDiGraph()
for node in node_ids:
    H.add_node(
        f"{node[0]:.{COORD_PRECISION}f},{node[1]:.{COORD_PRECISION}f}",
        x=float(node[0]),
        y=float(node[1]),
        node_idx=int(node_to_idx[node]),
    )

for u, v, key, data in G.edges(keys=True, data=True):
    H.add_edge(
        f"{u[0]:.{COORD_PRECISION}f},{u[1]:.{COORD_PRECISION}f}",
        f"{v[0]:.{COORD_PRECISION}f},{v[1]:.{COORD_PRECISION}f}",
        key=int(key),
        zone=str(data.get("zone", "")),
        source_file=str(data.get("source_file", "")),
        source_feature_id=str(data.get("source_feature_id", "")),
        length_crs_units=float(data.get("length_crs_units", 0.0)),
        maxspeed=str(data.get("maxspeed", "")),
        oneway_source=str(data.get("oneway_source", "")),
    )

nx.write_graphml(H, OUT_DIR / "road_graph.graphml")

summary = {
    "seed": SEED,
    "zones_requested": len(ZONES),
    "zones_processed": int(
        (audit_df["status"] == "PROCESSED").sum()
    ) if not audit_df.empty else 0,
    "nodes": G.number_of_nodes(),
    "directed_edges": G.number_of_edges(),
    "invalid_geometries_seen": invalid_geometry_count,
    "skipped_geometries": skipped_geometry_count,
    "coordinate_precision": COORD_PRECISION,
    "output_dir": str(OUT_DIR),
    "warning": (
        "This is a reconstructed graph, not proof of identical "
        "training graph/node ordering. Do not use its indices with "
        "the checkpoint unless the saved sampled_nodes mapping matches."
    ),
}

with open(OUT_DIR / "graph_build_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

display(audit_df)
print(json.dumps(summary, indent=2))
print("\nGraph rebuild finished.")

Project: C:\Users\prana\Downloads\capstone_final_draft
Data: C:\Users\prana\Downloads\capstone_final_draft\data
Output: C:\Users\prana\Downloads\capstone_final_draft\outputs\graph_rebuilt

Processing Central: central-zone.gpkg
  Layer: gis_osm_roads_free | Features: 2116520 | Nodes: 3657408 | Directed edges: 4225578

Processing Eastern: eastern-zone.gpkg
  Layer: gis_osm_roads_free | Features: 1526546 | Nodes: 6287589 | Directed edges: 7274772

Processing North-Eastern: north-eastern-zone.gpkg
  Layer: gis_osm_roads_free | Features: 304952 | Nodes: 6808805 | Directed edges: 7884017

Processing Northern: northern-zone.gpkg
  Layer: gis_osm_roads_free | Features: 1896174 | Nodes: 10045455 | Directed edges: 11664543

Processing Southern: southern-zone.gpkg
  Layer: gis_osm_roads_free | Features: 3410375 | Nodes: 15754196 | Directed edges: 18470436

Processing Western: western-zone.gpkg
  Layer: gis_osm_railways_free | Features: 17192 | Nodes: 15777293 | Directed edges: 18504808


In [ ]:
# ============================================================
# DETAILED ROAD MAPS FOR ALL SIX ZONES
# Source: original GeoPackages in data/
# Output: outputs/detailed_road_maps/
# ============================================================

from pathlib import Path
import math
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import pandas as pd
import pyogrio

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "detailed_road_maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ZONES = {
    "Central": "central-zone.gpkg",
    "Eastern": "eastern-zone.gpkg",
    "North-Eastern": "north-eastern-zone.gpkg",
    "Northern": "northern-zone.gpkg",
    "Southern": "southern-zone.gpkg",
    "Western": "western-zone.gpkg",
}

# Process large layers in chunks to limit memory usage.
CHUNK_SIZE = 50_000

# Plot limits: increase for more detail, at the cost of runtime.
MAX_FEATURES_PER_ZONE = 500_000

# Distinguish roads by their highway classification.
ROAD_STYLES = {
    "motorway":   ("#e63946", 2.0),
    "trunk":      ("#f4a261", 1.7),
    "primary":    ("#ffca3a", 1.35),
    "secondary":  ("#8ac926", 1.0),
    "tertiary":   ("#38bdf8", 0.75),
    "residential":("#e2e8f0", 0.48),
    "service":    ("#94a3b8", 0.30),
    "unclassified":("#cbd5e1", 0.42),
    "other":      ("#cbd5e1", 0.45),
}

def road_class(value):
    if value is None or pd.isna(value):
        return "other"

    # OSM data may contain lists or semicolon-separated classes.
    value = str(value).lower().split(";")[0].strip()

    return value if value in ROAD_STYLES else "other"


def line_parts(geometry):
    """Return individual line geometries without straightening curves."""
    if geometry is None or geometry.is_empty:
        return []

    if geometry.geom_type == "LineString":
        return [geometry]

    if geometry.geom_type == "MultiLineString":
        return list(geometry.geoms)

    if geometry.geom_type == "GeometryCollection":
        result = []
        for geom in geometry.geoms:
            result.extend(line_parts(geom))
        return result

    return []


def find_road_layer(gpkg):
    """Find a layer with line geometry."""
    layers = gpd.list_layers(gpkg)

    for _, info in layers.iterrows():
        layer_name = str(info["name"])
        geometry_type = str(info.get("geometry_type", "")).lower()

        if "line" in geometry_type:
            return layer_name

    # Fallback: inspect a small sample from each layer.
    for _, info in layers.iterrows():
        layer_name = str(info["name"])
        try:
            sample = gpd.read_file(gpkg, layer=layer_name, rows=5)
            if (
                not sample.empty
                and sample.geometry.geom_type.isin(
                    ["LineString", "MultiLineString"]
                ).any()
            ):
                return layer_name
        except Exception:
            continue

    return None


def render_zone(zone, filename):
    gpkg = DATA_DIR / filename

    if not gpkg.exists():
        print(f"SKIP {zone}: missing file {gpkg}")
        return

    layer = find_road_layer(gpkg)

    if layer is None:
        print(f"SKIP {zone}: no road line layer found")
        return

    info = pyogrio.read_info(gpkg, layer=layer)
    feature_count = info.get("features", -1)
    source_crs = info.get("crs")

    if not source_crs:
        raise ValueError(
            f"{zone}: layer {layer} has no CRS; refusing to "
            "guess coordinate units."
        )

    total_limit = (
        MAX_FEATURES_PER_ZONE
        if feature_count is None or feature_count < 0
        else min(feature_count, MAX_FEATURES_PER_ZONE)
    )

    # Use a local metric projection for better geometry display.
    # EPSG:3857 is used here for consistent regional visualization,
    # not for accurate distance measurement.
    target_crs = "EPSG:3857"

    fig, ax = plt.subplots(figsize=(18, 16))
    fig.patch.set_facecolor("#0b1120")
    ax.set_facecolor("#0b1120")

    collections = {key: [] for key in ROAD_STYLES}
    processed = 0
    plotted = 0

    # Read sequential chunks rather than loading the entire layer.
    for offset in range(0, total_limit, CHUNK_SIZE):
        count = min(CHUNK_SIZE, total_limit - offset)

        roads = gpd.read_file(
            gpkg,
            layer=layer,
            engine="pyogrio",
            skip_features=offset,
            max_features=count,
        )

        if roads.empty:
            continue

        roads = roads[roads.geometry.notna()]
        roads = roads[~roads.geometry.is_empty]

        if roads.crs is None:
            raise ValueError(f"{zone}: chunk has no CRS")

        if str(roads.crs) != target_crs:
            roads = roads.to_crs(target_crs)

        highway_col = (
            "highway" if "highway" in roads.columns else None
        )

        for row in roads.itertuples(index=False):
            geom = getattr(row, roads.geometry.name)
            highway = (
                getattr(row, highway_col)
                if highway_col
                else None
            )
            category = road_class(highway)

            for part in line_parts(geom):
                coords = list(part.coords)

                if len(coords) < 2:
                    continue

                xy = [
                    (float(c[0]), float(c[1]))
                    for c in coords
                    if math.isfinite(float(c[0]))
                    and math.isfinite(float(c[1]))
                ]

                if len(xy) >= 2:
                    collections[category].append(xy)
                    plotted += 1

        processed += len(roads)
        print(
            f"{zone}: processed {processed:,}/{total_limit:,} "
            f"features; line parts={plotted:,}"
        )

        del roads

    # Draw lower-priority streets first and major roads last.
    draw_order = [
        "other", "service", "unclassified", "residential",
        "tertiary", "secondary", "primary", "trunk", "motorway"
    ]

    for category in draw_order:
        segments = collections[category]

        if not segments:
            continue

        color, width = ROAD_STYLES[category]

        ax.add_collection(
            LineCollection(
                segments,
                colors=color,
                linewidths=width,
                alpha=0.95,
                capstyle="round",
                joinstyle="round",
                rasterized=True,
            )
        )

    ax.autoscale()
    ax.margins(0.025)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    ax.set_title(
        f"{zone} Road Network\n"
        f"Source layer: {layer} | Features processed: {processed:,}",
        color="white",
        fontsize=18,
        pad=18,
    )

    legend_items = [
        Line2D(
            [0], [0],
            color=ROAD_STYLES[category][0],
            linewidth=ROAD_STYLES[category][1],
            label=category.title(),
        )
        for category in [
            "motorway", "trunk", "primary", "secondary",
            "tertiary", "residential", "service"
        ]
    ]

    legend = ax.legend(
        handles=legend_items,
        loc="lower left",
        frameon=True,
        fontsize=10,
        facecolor="#111827",
        edgecolor="#475569",
    )

    for text in legend.get_texts():
        text.set_color("white")

    output_path = OUT_DIR / (
        zone.lower().replace("-", "_").replace(" ", "_")
        + "_detailed.png"
    )

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.close(fig)

    print(f"SAVED: {output_path}")
    print(f"  Source features processed: {processed:,}")
    print(f"  Line parts plotted: {plotted:,}")


# ---------- Generate all six maps ----------
for zone, filename in ZONES.items():
    print(f"\n{'=' * 65}\nZONE: {zone}\n{'=' * 65}")

    try:
        render_zone(zone, filename)
    except Exception as exc:
        print(f"ERROR in {zone}: {type(exc).__name__}: {exc}")

print("\nFinished. Maps saved in:", OUT_DIR)


ZONE: Western
Western: processed 17,192/17,192 features; line parts=17,192
SAVED: C:\Users\prana\Downloads\capstone_final_draft\outputs\detailed_road_maps\western_detailed.png
  Source features processed: 17,192
  Line parts plotted: 17,192

Finished. Maps saved in: C:\Users\prana\Downloads\capstone_final_draft\outputs\detailed_road_maps
